# 🛡️ SAFE-VISION: Qwen-VL Google Colab Inference Server
**Hardware Target**: Google Colab with NVIDIA GPU (T4 free or A100)
**Precision**: Native `torch.bfloat16` / `torch.float16`
**Purpose**: High-speed cloud GPU accelerator for the SafeAd Brand Owner Workspace frontend.

### Quick Setup Steps:
1. Go to **Runtime -> Change runtime type** and select **GPU** (T4 or A100).
2. Run **Cell 1** to verify your GPU.
3. Run **Cell 2** to install dependencies.
4. Run **Cell 3** to launch the Qwen-VL API server and generate your public ngrok tunnel URL.
5. Copy the generated **Public URL** (e.g. `https://xxxx.ngrok-free.app`) and paste it into the **SafeAd Streamlit Frontend** under *Google Colab Qwen3-VL Server Settings*!

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Install High-Performance Inference Dependencies
!pip install -q -U "transformers>=4.48.0" accelerate qwen-vl-utils fastapi uvicorn python-multipart pyngrok pillow opencv-python nest-asyncio

In [ ]:
# Cell 3: Launch SafeAd Qwen-VL Server & Expose via ngrok
import os, sys, time, json, tempfile
from PIL import Image
import torch
from transformers import AutoProcessor

# Universal Vision-Language loader for transformers 4.45+
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as QwenVLClass
except ImportError:
    try:
        from transformers import AutoModelForImageTextToText as QwenVLClass
    except ImportError:
        from transformers import Qwen2VLForConditionalGeneration as QwenVLClass

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import nest_asyncio
nest_asyncio.apply()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f'[SAFE-VISION] Target Device: {device} | Precision: {dtype}')

# Model identifier: Qwen2.5-VL-7B-Instruct (latest official production release)
MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'
print(f'[SAFE-VISION] Loading {MODEL_ID} weights...')
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = QwenVLClass.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map='auto' if device == 'cuda' else None
    ).eval()
    print(f'[SUCCESS] Loaded {MODEL_ID} successfully!')
except Exception as e:
    print(f'[FALLBACK] Loading Qwen/Qwen2-VL-7B-Instruct: {e}')
    MODEL_ID = 'Qwen/Qwen2-VL-7B-Instruct'
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = QwenVLClass.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map='auto').eval()

try:
    from qwen_vl_utils import process_vision_info
except ImportError:
    process_vision_info = None

app = FastAPI(title='SafeAd Qwen-VL Server')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

SYSTEM_PROMPT = '''You are a senior digital advertising trust and safety compliance officer.
Analyze the provided advertisement creative, extracted on-screen text (OCR), and speech audio transcript.
Evaluate the content against the four official statutory advertising policy tiers:
1. SAFE_FOR_ALL: General family audiences, consumer goods, education, technology. Zero adult themes, zero physical violence.
2. SAFE_14_PLUS: Mild adolescent action, competitive esports, teen cosmetics, dark fantasy. Strictly no explicit nudity, real combat violence, alcohol branding, or gambling.
3. SAFE_18_PLUS: Lawful commercial goods legally restricted to adults (alcoholic beverages, licensed casino wagering, adult dating, high-risk finance). Must contain statutory warnings.
4. UNSAFE_FOR_ALL: Prohibited contraband, real physical violence, hate speech, financial scams ("paisa double", guaranteed return crypto schemes), malware, phishing.

Respond STRICTLY in valid JSON matching this schema:
{
  "reasoning": "<concise step-by-step policy justification>",
  "predicted_class": "<SAFE_FOR_ALL | SAFE_14_PLUS | SAFE_18_PLUS | UNSAFE_FOR_ALL>",
  "confidence_scores": {
    "SAFE_FOR_ALL": 0.0,
    "SAFE_14_PLUS": 0.0,
    "SAFE_18_PLUS": 0.0,
    "UNSAFE_FOR_ALL": 0.0
  },
  "statutory_risk_score": 0.0,
  "detected_violations": ["violation_tags"]
}'''

@app.get('/health')
def health():
    gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    return {
        'status': 'online',
        'model': MODEL_ID,
        'device': str(device),
        'gpu': gpu_info,
        'precision': str(dtype),
        'service': 'SAFE-VISION Qwen-VL Inference Engine'
    }

@app.post('/predict_qwen')
@app.post('/predict')
async def predict_qwen(
    file: UploadFile = File(...),
    title: str = Form(''),
    caption: str = Form(''),
    extracted_ocr: str = Form(''),
    audio_transcript: str = Form('')
):
    start_time = time.perf_counter()
    contents = await file.read()
    suffix = os.path.splitext(file.filename)[1].lower() if file.filename else '.jpg'
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(contents)
        tmp_path = tmp.name

    try:
        if suffix in ['.mp4', '.mov', '.avi', '.webm', '.mkv']:
            import cv2
            cap = cv2.VideoCapture(tmp_path)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, total // 2))
            ret, frame = cap.read()
            cap.release()
            pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)) if ret else Image.new('RGB', (300, 300), (128, 128, 128))
        else:
            pil_img = Image.open(tmp_path).convert('RGB')

        user_text = f"Creative Title: '{title}'\nCaption / Ad Copy: '{caption}'\n"
        if extracted_ocr:
            user_text += f"Extracted OCR Text: '{extracted_ocr}'\n"
        if audio_transcript:
            user_text += f"Audio Speech Transcript: '{audio_transcript}'\n"
        user_text += 'Evaluate compliance and return strictly the JSON verdict.'

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': [{'type': 'image', 'image': pil_img}, {'type': 'text', 'text': user_text}]}
        ]
        prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        if process_vision_info:
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(text=[prompt_text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(device)
        else:
            inputs = processor(text=[prompt_text], images=[pil_img], return_tensors='pt').to(device)

        with torch.inference_mode():
            generated_ids = model.generate(**inputs, max_new_tokens=400, do_sample=False)

        trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
        raw_resp = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        cleaned = raw_resp
        if '```json' in cleaned:
            cleaned = cleaned.split('```json')[-1].split('```')[0].strip()
        elif '```' in cleaned:
            cleaned = cleaned.split('```')[-1].split('```')[0].strip()

        try:
            parsed = json.loads(cleaned)
        except Exception:
            import re
            m = re.search(r'\{.*\}', cleaned, re.DOTALL)
            parsed = json.loads(m.group(0)) if m else {'predicted_class': 'SAFE_FOR_ALL', 'reasoning': cleaned, 'statutory_risk_score': 10.0}

        pred_class = parsed.get('predicted_class', 'SAFE_FOR_ALL')
        conf_scores = parsed.get('confidence_scores', {})
        confidence = float(conf_scores.get(pred_class, 0.85))
        risk_score = float(parsed.get('statutory_risk_score', 15.0))
        reasoning = parsed.get('reasoning', 'Evaluation complete.')
        elapsed = round(time.perf_counter() - start_time, 3)

        return {
            'classification': pred_class,
            'predicted_class': pred_class,
            'confidence': confidence,
            'confidence_scores': conf_scores,
            'risk_score': risk_score,
            'reasoning': reasoning,
            'explanation': f'[{MODEL_ID} GPU]: {reasoning}',
            'violations': parsed.get('detected_violations', []),
            'model_version': f'{MODEL_ID} (Google Colab {str(device).upper()})',
            'latency_seconds': elapsed
        }
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

# Expose via ngrok
from pyngrok import ngrok
# Paste your free ngrok auth token if you have one
NGROK_TOKEN = ''  # <-- Optional: Paste your free ngrok auth token
if NGROK_TOKEN.strip():
    try:
        ngrok.set_auth_token(NGROK_TOKEN.strip())
    except Exception:
        pass
# Check for existing active tunnel or create new
active_tunnels = ngrok.get_tunnels()
if active_tunnels:
    public_tunnel = active_tunnels[0]
else:
    public_tunnel = ngrok.connect(8000)
print('=' * 75)
print(f'🚀 PUBLIC GOOGLE COLAB TUNNEL URL: {public_tunnel.public_url}')
print('👉 Copy this URL into the Streamlit Brand Owner Workspace Colab field!')
print('=' * 75)

# Notebook-safe server launch (attaches directly to Colab event loop)
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
await server.serve()
